In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import kruskal

folder_name = "borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016"

metadata = pd.read_csv(f"{folder_name}/metadata.tsv", sep="\t")
mtb = pd.read_csv(f"{folder_name}/mtb.tsv", sep="\t")

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = metadata.merge(mtb, on="Sample")[feature_cols]
y = metadata["Study.Group"]

print("X shape:", X.shape)
print(y.value_counts())

X shape: (90, 4626)
Study.Group
Normal    54
CD        26
UC        10
Name: count, dtype: int64


In [2]:
# Sparsity filter (>80% zeros removed) — same as before
sparsity = (X == 0).sum(axis=0) / len(X) * 100
X_filtered = X[sparsity[sparsity <= 80].index]
print("Features after sparsity filter:", X_filtered.shape[1])

# Outlier detection (Isolation Forest, same as before)
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
outlier_flags = iso.fit_predict(X_filtered)
X_clean_ibd = X_filtered[outlier_flags == 1].reset_index(drop=True)
y_clean_ibd = y[outlier_flags == 1].reset_index(drop=True)
print("Samples after outlier removal:", len(X_clean_ibd))

# IBDpred style: log2 transform (not CLR — this is IBDpred's own preprocessing choice)
X_log = np.log2(X_clean_ibd + 1)

# Kruskal-Wallis feature selection (top 40)
def kruskal_top_k(Xd, yd, k):
    groups = pd.Series(yd).unique()
    pvals = {c: kruskal(*[Xd.loc[pd.Series(yd).values == g, c] for g in groups])[1] for c in Xd.columns}
    return sorted(pvals, key=pvals.get)[:k]

train_idx3, test_idx3 = train_test_split(np.arange(len(y_clean_ibd)), test_size=0.2, stratify=y_clean_ibd, random_state=42)

top40_jacobs = kruskal_top_k(X_log.iloc[train_idx3], y_clean_ibd.iloc[train_idx3], 40)
print("Top 40 features selected")

Features after sparsity filter: 2342
Samples after outlier removal: 85
Top 40 features selected


In [3]:
from sklearn.linear_model import LogisticRegression

X_train_ibd = X_log[top40_jacobs].iloc[train_idx3]
X_test_ibd = X_log[top40_jacobs].iloc[test_idx3]

le_ibd = LabelEncoder()
y_ibd_enc = le_ibd.fit_transform(y_clean_ibd)
y_train_ibd, y_test_ibd = y_ibd_enc[train_idx3], y_ibd_enc[test_idx3]

# Random Forest
rf_ibdpred = RandomForestClassifier(n_estimators=500, max_features="sqrt", random_state=42)
rf_ibdpred.fit(X_train_ibd, y_train_ibd)
rf_ibd_probs = rf_ibdpred.predict_proba(X_test_ibd)
rf_ibd_auc = roc_auc_score(y_test_ibd, rf_ibd_probs, multi_class="ovr", average="macro")

# Elastic Net
from sklearn.preprocessing import StandardScaler
X_train_ibd_scaled = StandardScaler().fit_transform(X_train_ibd)
X_test_ibd_scaled = StandardScaler().fit(X_train_ibd).transform(X_test_ibd)

en_ibdpred = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1, max_iter=20000, random_state=42)
en_ibdpred.fit(X_train_ibd_scaled, y_train_ibd)
en_ibd_probs = en_ibdpred.predict_proba(X_test_ibd_scaled)
en_ibd_auc = roc_auc_score(y_test_ibd, en_ibd_probs, multi_class="ovr", average="macro")

print(f"IBDpred-style Random Forest (k=40) — macro-AUC: {rf_ibd_auc:.3f}")
print(f"IBDpred-style Elastic Net (k=40) — macro-AUC: {en_ibd_auc:.3f}")

IBDpred-style Random Forest (k=40) — macro-AUC: 0.843
IBDpred-style Elastic Net (k=40) — macro-AUC: 0.794
